# Pertemuan 10 - Algoritma Klasifikasi (Bagian 2)

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Mata Kuliah:** Pengantar Data Science  
**Topik:** Random Forest, Ensemble Learning, dan Imbalanced Dataset

Notebook ini dibuat untuk praktikum Pertemuan 10. Studi kasus yang digunakan adalah **Customer Churn**, yaitu prediksi pelanggan yang berisiko berhenti menggunakan layanan. Model utama yang dipakai adalah **Random Forest Classifier** dengan evaluasi yang memperhatikan data tidak seimbang.


## 1. Import Library

Library yang digunakan meliputi Pandas dan NumPy untuk pengolahan data, Matplotlib dan Seaborn untuk visualisasi, serta Scikit-Learn untuk pemodelan Random Forest dan evaluasi klasifikasi.


In [ ]:
# Jalankan notebook ini di Google Colab atau Jupyter Notebook.
# Jika ada library yang belum tersedia, hapus tanda pagar pada baris berikut:
# !pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)


## 2. Load Dataset Customer Churn

Notebook ini akan mencoba membaca file lokal `telco_churn.csv`. Jika file tersebut belum ada, notebook akan mengambil dataset publik Telco Customer Churn dari GitHub. Target yang digunakan adalah kolom `Churn`, dengan nilai:

- `No` = pelanggan tidak churn
- `Yes` = pelanggan churn


In [ ]:
local_path = Path("telco_churn.csv")
url = "https://raw.githubusercontent.com/blastchar/telco-customer-churn/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

if local_path.exists():
    df = pd.read_csv(local_path)
    print("Dataset dibaca dari file lokal:", local_path)
else:
    df = pd.read_csv(url)
    print("Dataset dibaca dari URL publik.")

print("Shape dataset:", df.shape)
display(df.head())


## 3. Eksplorasi Data Awal

Pada tahap ini saya mengecek tipe data, missing value, dan proporsi kelas churn. Tujuannya untuk memastikan apakah dataset termasuk imbalanced dataset.


In [ ]:
print("Informasi dataset:")
df.info()

print("\nJumlah missing value per kolom:")
display(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDistribusi target Churn:")
display(df["Churn"].value_counts())
display((df["Churn"].value_counts(normalize=True) * 100).round(2))


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", palette=["#1b9e77", "#d95f02"])
plt.title("Distribusi Target Churn")
plt.xlabel("Churn")
plt.ylabel("Jumlah Pelanggan")
plt.show()


**Catatan EDA:** Jika proporsi pelanggan `Churn = Yes` jauh lebih kecil dibanding `No`, maka dataset ini termasuk tidak seimbang. Pada kondisi seperti ini, akurasi saja tidak cukup karena model bisa terlihat bagus walaupun gagal menangkap pelanggan yang benar-benar churn.


## 4. Data Cleaning dan Preprocessing

Beberapa langkah preprocessing yang dilakukan:

- Menghapus `customerID` dari fitur model karena hanya berisi identitas pelanggan.
- Mengubah `TotalCharges` menjadi numerik karena sering terbaca sebagai teks.
- Mengisi nilai kosong pada `TotalCharges` dengan median.
- Mengubah target `Churn` menjadi angka, yaitu `Yes = 1` dan `No = 0`.
- Melakukan encoding fitur kategorikal menggunakan `pd.get_dummies()`.


In [ ]:
df_clean = df.copy()

if "customerID" in df_clean.columns:
    customer_id = df_clean["customerID"].copy()
else:
    customer_id = pd.Series(np.arange(len(df_clean)), name="customerID")

df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(df_clean["TotalCharges"].median())

y = df_clean["Churn"].map({"No": 0, "Yes": 1})

X = df_clean.drop(columns=["Churn"])
if "customerID" in X.columns:
    X = X.drop(columns=["customerID"])

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

print("Shape fitur setelah encoding:", X_encoded.shape)
print("Distribusi target numerik:")
print(y.value_counts(normalize=True).round(3))

display(X_encoded.head())


## 5. Train-Test Split

Data dibagi menjadi data latih dan data uji dengan rasio 80:20. Parameter `stratify=y` digunakan agar proporsi churn pada data latih dan uji tetap sama seperti data asli.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Jumlah data latih:", X_tr.shape[0])
print("Jumlah data uji  :", X_te.shape[0])
print("\nProporsi target pada data latih:")
print(y_tr.value_counts(normalize=True).round(3))
print("\nProporsi target pada data uji:")
print(y_te.value_counts(normalize=True).round(3))


## 6. Model Random Forest Tanpa Class Weight

Model pertama dibuat sebagai pembanding awal. Model ini belum diberi penanganan khusus untuk data tidak seimbang.


In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

rf_base.fit(X_tr, y_tr)

pred_base = rf_base.predict(X_te)
proba_base = rf_base.predict_proba(X_te)[:, 1]


## 7. Model Random Forest dengan Class Weight Balanced

Model kedua menggunakan `class_weight="balanced"` agar model memberikan perhatian lebih pada kelas minoritas, yaitu pelanggan yang churn.


In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

rf_balanced.fit(X_tr, y_tr)

pred_balanced = rf_balanced.predict(X_te)
proba_balanced = rf_balanced.predict_proba(X_te)[:, 1]


## 8. Evaluasi Model

Metrik yang digunakan adalah Accuracy, Precision, Recall, F1-Score, ROC-AUC, dan PR-AUC. Pada kasus customer churn, **Recall** penting karena perusahaan ingin menangkap sebanyak mungkin pelanggan yang berisiko berhenti.


In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred)
    return {
        "model": name,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1],
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }

results = pd.DataFrame([
    evaluate_model("RF Baseline", y_te, pred_base, proba_base),
    evaluate_model("RF Class Weight Balanced", y_te, pred_balanced, proba_balanced),
])

display(results.round(3))


In [ ]:
for name, pred, proba in [
    ("RF Baseline", pred_base, proba_base),
    ("RF Class Weight Balanced", pred_balanced, proba_balanced),
]:
    print(f"\n=== {name} ===")
    print("Confusion Matrix:")
    print(confusion_matrix(y_te, pred))
    print("\nClassification Report:")
    print(classification_report(y_te, pred, target_names=["Tidak Churn", "Churn"]))
    print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3))
    print("PR-AUC :", round(average_precision_score(y_te, proba), 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, pred) in zip(
    axes,
    [("RF Baseline", pred_base), ("RF Class Weight Balanced", pred_balanced)]
):
    cm = confusion_matrix(y_te, pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Tidak Churn", "Churn"],
        yticklabels=["Tidak Churn", "Churn"],
        ax=ax,
    )
    ax.set_title(name)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")

plt.tight_layout()
plt.show()


In [ ]:
metric_plot = results.melt(
    id_vars="model",
    value_vars=["accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"],
    var_name="metrik",
    value_name="nilai",
)

plt.figure(figsize=(11, 5))
sns.barplot(data=metric_plot, x="metrik", y="nilai", hue="model")
plt.ylim(0, 1.05)
plt.title("Perbandingan Metrik Random Forest")
plt.xlabel("Metrik")
plt.ylabel("Nilai")
plt.legend(title="Model")
plt.show()


## 9. Threshold Tuning

Threshold default untuk klasifikasi adalah 0.5. Pada data churn yang tidak seimbang, threshold dapat diturunkan agar model lebih banyak menangkap pelanggan yang berisiko churn. Dampaknya, recall biasanya naik tetapi precision bisa turun.


In [ ]:
threshold_rows = []

for threshold in [0.25, 0.30, 0.35, 0.40, 0.50]:
    pred_thr = (proba_balanced >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_te, pred_thr),
        "recall": recall_score(y_te, pred_thr),
        "f1_score": f1_score(y_te, pred_thr),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(3))


In [ ]:
plt.figure(figsize=(8, 5))
for metric in ["precision", "recall", "f1_score"]:
    plt.plot(threshold_df["threshold"], threshold_df[metric], marker="o", label=metric)

plt.title("Pengaruh Threshold terhadap Precision, Recall, dan F1-Score")
plt.xlabel("Threshold")
plt.ylabel("Nilai")
plt.ylim(0, 1)
plt.legend()
plt.show()


## 10. Feature Importance

Feature importance digunakan untuk melihat fitur yang paling berpengaruh terhadap prediksi churn pelanggan.


In [ ]:
importance_df = pd.DataFrame({
    "fitur": X_encoded.columns,
    "importance": rf_balanced.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(15).round(4))

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df.head(10), x="importance", y="fitur", palette="viridis")
plt.title("Top 10 Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Fitur")
plt.show()


## 11. Prediksi Probabilitas Churn

Bagian ini menampilkan pelanggan pada data uji yang memiliki probabilitas churn paling tinggi. Informasi seperti ini dapat membantu perusahaan membuat prioritas retensi pelanggan.


In [ ]:
risk_df = pd.DataFrame({
    "customerID": customer_id.loc[X_te.index].values,
    "actual_churn": y_te.map({0: "No", 1: "Yes"}).values,
    "probabilitas_churn": proba_balanced,
})

risk_df = risk_df.sort_values("probabilitas_churn", ascending=False)
display(risk_df.head(10))


## 12. Kesimpulan

Pada praktikum ini saya mempelajari bahwa Random Forest dapat digunakan untuk memprediksi customer churn dengan cukup baik karena model ini menggabungkan banyak decision tree. Dataset churn termasuk imbalanced dataset, sehingga evaluasi tidak boleh hanya mengandalkan accuracy. Recall menjadi metrik penting karena perusahaan perlu menangkap sebanyak mungkin pelanggan yang berisiko churn agar dapat diberikan tindakan retensi. Berdasarkan hasil evaluasi dan threshold tuning, model dapat disesuaikan tergantung kebutuhan perusahaan, apakah ingin lebih banyak menangkap pelanggan churn atau menjaga agar alarm palsu tidak terlalu banyak.


In [ ]:
best_model = results.sort_values(["recall", "f1_score"], ascending=False).iloc[0]

print("Kesimpulan otomatis:")
print(
    f"Model dengan recall tertinggi adalah {best_model['model']} "
    f"dengan recall {best_model['recall']:.3f} dan F1-score {best_model['f1_score']:.3f}."
)
print(
    "Untuk kasus customer churn, recall perlu diperhatikan karena pelanggan yang benar-benar "
    "berisiko churn sebaiknya tidak terlewat oleh model."
)
print(
    "Jika perusahaan ingin menangkap lebih banyak pelanggan berisiko, threshold dapat diturunkan, "
    "namun precision perlu tetap dipantau agar tindakan retensi tidak terlalu banyak salah sasaran."
)
